# Paradigm A: Short-Term Plasticity RNN — DMS, DRMS and ABBA

This notebook ports the **short-term synaptic plasticity (STP) RNN** from

> Masse, N. Y., Yang, G. R., Song, H. F., Wang, X. J. & Freedman, D. J.
> *Circuit mechanisms for the maintenance and manipulation of information in working memory.*
> Nature Neuroscience, 2019.

into the NeuralRNN framework. The paper asks two questions:

1. Can STP support **activity-silent manipulation** of working-memory (WM) information?
2. Can STP explain why different WM tasks evoke **different amounts of persistent activity**?

The central finding is that STP can maintain information silently during short delays,
but tasks that require **manipulation** of the memorandum force the network to develop
**persistent activity**, and the amount of persistent activity scales with the degree of
manipulation required.

We implement an inline STP-RNN model, three task datasets (DMS, DRMS, ABBA), and a custom
supervised objective that includes the L2 firing-rate penalty used in the paper. We then
compare the trained solutions and highlight the difference between *maintenance* (DMS) and
*manipulation* (DRMS / ABBA).

> **Note on the model**: The STP-RNN is defined inline in this notebook. Once the port is
> validated, it can be moved to `src/neuralrnn/models/stp_rnn/` and registered with
> `AutoModel`/`AutoConfig`, following the Contract-A pattern used by `CTRNNModel`.


## 0. Setup

In [ ]:
import sys
sys.path.append('../src')

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

from neuralrnn import (
    NeuralRNNConfig, NeuralDynamicsModel, register_model,
    Trainer, TrainingArguments, AutoConfig, AutoModel,
    BaseDataset, load_dataset,
)
from neuralrnn.train.objectives.base import Objective
from neuralrnn.analysis import fit_pca, find_fixed_points, compute_vector_field
from neuralrnn.analysis.linearization import linearize, dominant_direction
from neuralrnn.visualization import plot_trajectories_2d

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'
print(f'Device: {DEVICE}')

# Model / training constants taken from the paper (Table 1)
DT = 10.0          # ms
TAU = 100.0        # ms
N_MOTION_DIRS = 8
N_INPUT = 24       # 24 motion-tuned input neurons (3 per direction)
N_HIDDEN = 100     # 80 excitatory + 20 inhibitory
N_OUTPUT = 3       # fix, match, non-match
ALPHA = DT / TAU


## 1. Short-term synaptic plasticity: the model

In the paper each recurrent neuron is either **excitatory** (E, 80 neurons) or **inhibitory**
(I, 20 neurons). The recurrent weight matrix is decomposed as

$$W^{\text{rec}} = W^{\text{rec,+}} D$$

where $W^{\text{rec,+}}$ is trained and element-wise non-negative, and $D$ is a fixed
diagonal matrix with $+1$ for E units and $-1$ for I units. This enforces Dale's principle:
E neurons make only excitatory outgoing connections, I neurons make only inhibitory ones.

The recurrent dynamics follow a continuous-time RNN with ReLU activation.  The reference
TensorFlow code discretizes it as

$$\mathbf{r}_t = f\left((1-\alpha)\mathbf{r}_{t-1} + \alpha\left(W^{\text{rec}} \mathbf{h}^{\text{post}}_{t-1} + W^{\text{in}} \mathbf{u}_t + \mathbf{b}^{\text{rec}}\right) + \sqrt{2\alpha}\sigma_{\text{rec}} N(0,1)\right),$$

where $f$ is ReLU, $\alpha = \Delta t / \tau = 0.1$, and the postsynaptic input uses the
*STP-modulated* presynaptic activity:

$$\mathbf{h}^{\text{post}}_{t-1} = \mathbf{x}_{t-1} \, \mathbf{u}_{t-1} \, \mathbf{r}_{t-1}.$$

Several architectural details are essential for reproducing the paper's results and are
often omitted in simplified ports:

1. **ReLU on input/output weights.**  The reference uses `relu(w_in)` and `relu(w_out)` so
   that input projections and readout weights are non-negative.  This reflects the fact
   that the input layer and long-range readout are excitatory.
2. **No self-connections.**  The recurrent weight diagonal is masked to zero, preventing
   units from connecting to themselves.  This removes a simple persistent-activity solution
   and forces the network to rely on STP or inter-neuron dynamics.
3. **Excitatory-only readout.**  Output weights from the inhibitory subpopulation are zero,
   again consistent with Dale's principle for long-range projections.
4. **STP steady-state initialization.**  The utilization variable $u$ must start at its
   asymptotic value $U$ (0.15 for facilitating, 0.45 for depressing).  Starting at $u=1.0$
   destroys the low-baseline utilization that lets synaptic efficacy encode the sample.
5. **Noise scaling.**  Recurrent noise has standard deviation $\sqrt{2\alpha}\sigma_{\text{rec}}$
   and is added to the blended update before the final ReLU.  Input noise is scaled by
   $\sqrt{2/\alpha}$.

The STP variables evolve according to Mongillo et al. (2008):

$$\frac{dx}{dt} = \frac{1-x}{\tau_x} - u x r \Delta t, \qquad
\frac{du}{dt} = \frac{U-u}{\tau_u} + U(1-u) r \Delta t.$$

For **facilitating** synapses: $\tau_x=1500$ ms, $\tau_u=200$ ms, $U=0.15$.
For **depressing** synapses: $\tau_x=200$ ms, $\tau_u=1500$ ms, $U=0.45$.

The paper uses a "full" configuration in which half the neurons are facilitating and half
are depressing (alternating along the hidden dimension).

The readout is a linear projection to $N_{\text{out}}=3$ units, followed by softmax during
training (cross-entropy loss).

The total training loss is

$$\mathcal{L} = \underbrace{\text{CrossEntropy}(\hat{\mathbf{y}}, \mathbf{y}^{\text{target}})}_{\text{task loss, masked}} + \beta \underbrace{\frac{1}{N_{\text{rec}}}\sum_i r_i^2}_{\text{L2 firing-rate penalty}} + \text{weight cost} \underbrace{\frac{1}{N_{\text{rec}}^2}\sum_{ij}(W^{\text{rec,+}}_{ij})^2}_{\text{weight penalty}}$$

In the paper $\beta = 0.02$; the weight penalty is $0$ by default.

## 2. Inline STP-RNN model

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
class STPRNNConfig(NeuralRNNConfig):
    """Configuration for the STP-RNN of Masse et al. 2019."""
    model_type = "stp_rnn"

    def __init__(
        self,
        input_dim: int = N_INPUT,
        latent_dim: int = N_HIDDEN,
        output_dim: int = N_OUTPUT,
        dt: float = DT,
        tau: float = TAU,
        activation: str = "relu",
        ei_ratio: float = 0.8,
        sigma_rec: float = 0.5,
        sigma_in: float = 0.1,
        tau_x_fac: float = 1500.0,
        tau_x_dep: float = 200.0,
        tau_u_fac: float = 200.0,
        tau_u_dep: float = 1500.0,
        U_fac: float = 0.15,
        U_dep: float = 0.45,
        spike_cost: float = 0.02,
        weight_cost: float = 0.0,
        synapse_config: str = "full",  # 'full' = alternating fac/dep
        gamma_shape_exc: float = 0.1,
        gamma_shape_inh: float = 0.2,
        gamma_scale: float = 1.0,
        **kwargs,
    ):
        super().__init__(
            input_dim=input_dim, latent_dim=latent_dim, output_dim=output_dim,
            dt=dt, activation=activation, **kwargs,
        )
        self.tau = tau
        self.ei_ratio = ei_ratio
        self.sigma_rec = sigma_rec
        self.sigma_in = sigma_in
        self.tau_x_fac = tau_x_fac
        self.tau_x_dep = tau_x_dep
        self.tau_u_fac = tau_u_fac
        self.tau_u_dep = tau_u_dep
        self.U_fac = U_fac
        self.U_dep = U_dep
        self.spike_cost = spike_cost
        self.weight_cost = weight_cost
        self.synapse_config = synapse_config
        self.gamma_shape_exc = gamma_shape_exc
        self.gamma_shape_inh = gamma_shape_inh
        self.gamma_scale = gamma_scale


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
@register_model("stp_rnn")
class STPRNNModel(NeuralDynamicsModel):
    """STP-RNN with E/I separation and per-neuron short-term plasticity.

    This implementation follows the TensorFlow reference (Masse et al., 2019) as
    closely as possible:

    * Dale's principle: recurrent weights are |W| @ diag(E/I signs), so E units
      have non-negative outgoing weights and I units have non-positive weights.
    * No self-connections: the recurrent weight diagonal is masked to zero.
    * Excitatory-only readout: output weights from inhibitory neurons are zero.
    * Non-negative input/output weights: ReLU is applied to w_in and w_out.
    * STP steady state: initial utilization u0 equals the asymptotic value U for
      each synapse type (0.15 facilitating, 0.45 depressing), not 1.0.
    * Initial firing rate h0 = 0.1, matching the reference parameters.
    * Recurrent noise: sqrt(2*alpha)*sigma_rec, added to the blended update
      before the final ReLU (not to the pre-activation).
    """
    config_class = STPRNNConfig

    def __init__(self, config: STPRNNConfig):
        super().__init__(config)
        M = config.latent_dim
        self.alpha = 1.0 if config.dt is None else config.dt / config.tau
        self.act = {"relu": torch.relu, "tanh": torch.tanh}[config.activation]

        # Input / recurrent / output layers
        self.input2h = nn.Linear(config.input_dim, M, bias=False)
        self.h2h = nn.Linear(M, M, bias=False)
        self.bias_rnn = nn.Parameter(torch.zeros(M))
        self.readout_layer = nn.Linear(M, config.output_dim)

        # Dale sign mask: first e_size neurons are E (+1), rest I (-1)
        e_size = int(round(M * config.ei_ratio))
        sign = torch.ones(M)
        sign[e_size:] = -1.0
        self.register_buffer("dale_mask", torch.diag(sign))
        self.e_size = e_size

        # STP constants per neuron (computed once at init)
        alpha_x = torch.zeros(M)
        alpha_u = torch.zeros(M)
        U = torch.zeros(M)
        if config.synapse_config == "full":
            # alternating facilitating / depressing
            is_fac = torch.arange(M) % 2 == 0
        else:
            raise ValueError(f"Unknown synapse_config: {config.synapse_config}")

        alpha_x[is_fac] = config.dt / config.tau_x_fac
        alpha_u[is_fac] = config.dt / config.tau_u_fac
        U[is_fac] = config.U_fac
        alpha_x[~is_fac] = config.dt / config.tau_x_dep
        alpha_u[~is_fac] = config.dt / config.tau_u_dep
        U[~is_fac] = config.U_dep
        self.register_buffer("alpha_x", alpha_x)
        self.register_buffer("alpha_u", alpha_u)
        self.register_buffer("U", U)

        # Masks: no self-connections in recurrent weights; readout only from E units.
        w_rnn_mask = torch.ones(M, M) - torch.eye(M)
        self.register_buffer("w_rnn_mask", w_rnn_mask)
        w_out_mask = torch.ones(config.output_dim, M)
        w_out_mask[:, e_size:] = 0.0
        self.register_buffer("w_out_mask", w_out_mask)

        # Initial values.  Crucial: u0 must equal U for each synapse type so the
        # network starts from the correct STP steady state; u0=1.0 destroys the
        # low-baseline utilization that allows synaptic efficacy to encode the
        # sample direction.
        self.register_buffer("syn_x_init", torch.ones(M))
        syn_u_init = torch.full((M,), 0.3)
        syn_u_init[is_fac] = config.U_fac
        syn_u_init[~is_fac] = config.U_dep
        self.register_buffer("syn_u_init", syn_u_init)
        self.register_buffer("h0", torch.full((M,), 0.1))

        self._init_weights()
        self.apply_freeze_config()

    def _init_weights(self):
        """Initialize with the paper's gamma distribution (Table 1 / Methods)."""
        cfg = self.config
        M = cfg.latent_dim
        e_size = self.e_size
        with torch.no_grad():
            # Recurrent weights.  The reference first samples from Gamma(0.1, 1),
            # then boosts weights to/from inhibitory neurons to Gamma(0.2, 1) when
            # balance_EI is True.
            w_rnn = np.zeros((M, M), dtype=np.float32)
            w_rnn[:, :e_size] = np.random.gamma(cfg.gamma_shape_exc, cfg.gamma_scale,
                                                 size=(M, e_size))
            w_rnn[:, e_size:] = np.random.gamma(cfg.gamma_shape_inh, cfg.gamma_scale,
                                                 size=(M, M - e_size))
            # Zero diagonal to enforce "no self-connections".
            w_rnn *= (1.0 - np.eye(M, dtype=np.float32))
            self.h2h.weight.copy_(torch.from_numpy(w_rnn))

            # Input weights.  In the reference all input projections are
            # excitatory and sampled from Gamma(0.2, 1).
            w_in = np.random.gamma(0.2, cfg.gamma_scale,
                                   size=(M, cfg.input_dim)).astype(np.float32)
            self.input2h.weight.copy_(torch.from_numpy(w_in))

            # Output weights: excitatory-only readout (Gamma(0.1, 1)).
            w_out = np.zeros((cfg.output_dim, M), dtype=np.float32)
            w_out[:, :e_size] = np.random.gamma(cfg.gamma_shape_exc, cfg.gamma_scale,
                                                 size=(cfg.output_dim, e_size))
            self.readout_layer.weight.copy_(torch.from_numpy(w_out))

    def _recurrent_weight(self) -> torch.Tensor:
        """Dale-constrained recurrent weights: |W| * (1 - I) @ diag(signs)."""
        W = self.h2h.weight.abs() * self.w_rnn_mask
        return W @ self.dale_mask

    # ---------------- helper: STP update ----------------
    def _stp_step(self, r: torch.Tensor, x: torch.Tensor, u: torch.Tensor):
        """Advance STP variables given presynaptic firing rate r.

        Follows Mongillo et al. (2008) and the reference parameters.py:
            x <- x + alpha_x*(1 - x) - dt_sec * u*x*r
            u <- u + alpha_u*(U - u) + dt_sec * U*(1 - u)*r
        Both x and u are clipped to [0, 1].
        """
        x_new = x + self.alpha_x * (1.0 - x) - self.dt_sec * u * x * r
        u_new = u + self.alpha_u * (self.U - u) + self.dt_sec * self.U * (1.0 - u) * r
        x_new = torch.clamp(x_new, min=0.0, max=1.0)
        u_new = torch.clamp(u_new, min=0.0, max=1.0)
        return x_new, u_new

    @property
    def dt_sec(self):
        """Time step in seconds (used in STP equations)."""
        return self.config.dt / 1000.0

    def _freeze_groups(self) -> dict[str, list[str]]:
        return {
            "input": [r"^input2h\."],
            "recurrent": [r"^h2h\.", r"^bias_rnn$"],
            "output": [r"^readout_layer\."],
            "h0": [r"^h0$"],
        }

    # ---------------- hard contract (framework compatible) ----------------
    def recurrence(self, x_t, z_prev, *, inputs=None):
        """Single-step transition.

        z_prev can be:
          - a tensor of shape (B, 3M) -> concatenated [h, x, u]; used by analysis tools.
          - a tensor of shape (B, M) -> hidden state only (STP variables assumed = 1).
        """
        M = self.config.latent_dim
        if z_prev.shape[-1] == 3 * M:
            h, syn_x, syn_u = z_prev.split(M, dim=-1)
        elif z_prev.shape[-1] == M:
            h = z_prev
            syn_x = torch.ones_like(h)
            syn_u = torch.ones_like(h)
        else:
            raise ValueError(f"z_prev has unexpected dim {z_prev.shape[-1]}")

        # STP update: presynaptic activity h modulates x and u.
        syn_x_new, syn_u_new = self._stp_step(h, syn_x, syn_u)
        h_post = syn_u_new * syn_x_new * h

        # Input drive uses non-negative weights (excitatory projections only).
        input_drive = F.linear(x_t, F.relu(self.input2h.weight))
        # Recurrent drive uses Dale-constrained, no-self-connection weights.
        recurrent_drive = F.linear(h_post, self._recurrent_weight(), self.bias_rnn)
        pre = input_drive + recurrent_drive

        # Euler update.  The reference adds recurrent noise to the blended term
        # *before* the final ReLU, with magnitude sqrt(2*alpha)*sigma_rec.
        update = (1 - self.alpha) * h + self.alpha * pre
        if self.config.sigma_rec > 0 and self.training:
            noise_std = (2 * self.alpha) ** 0.5 * self.config.sigma_rec
            update = update + noise_std * torch.randn_like(update)
        h_new = self.act(update)

        if z_prev.shape[-1] == 3 * M:
            return torch.cat([h_new, syn_x_new, syn_u_new], dim=-1)
        return h_new

    def readout(self, z_t: torch.Tensor) -> torch.Tensor:
        """Readout from the firing-rate component of z_t.

        Output weights are ReLU-constrained and masked so that only excitatory
        neurons project to the readout, matching Dale's principle for long-range
        projections.
        """
        M = self.config.latent_dim
        if z_t.shape[-1] == 3 * M:
            h = z_t[..., :M]
        else:
            h = z_t
        w_out = F.relu(self.readout_layer.weight) * self.w_out_mask
        return F.linear(h, w_out, self.readout_layer.bias)

    # ---------------- framework helpers ----------------
    def init_state(self, batch_size: int, device="cpu") -> torch.Tensor:
        """Initial concatenated state [h0, x0, u0]."""
        h = self.h0.to(device).expand(batch_size, -1)
        sx = self.syn_x_init.to(device).expand(batch_size, -1)
        su = self.syn_u_init.to(device).expand(batch_size, -1)
        return torch.cat([h, sx, su], dim=-1)

    def forward(self, inputs=None, *, initial_state=None, n_steps=None, return_states=True):
        """Override forward to also return synaptic efficacies in extras."""
        if inputs is not None:
            assert inputs.dim() == 3, "inputs must be (batch, T, input_dim)"
            batch_size, T = inputs.shape[0], inputs.shape[1]
            device = inputs.device
        else:
            assert n_steps is not None and initial_state is not None
            batch_size, T, device = initial_state.shape[0], n_steps, initial_state.device

        z = initial_state if initial_state is not None else self.init_state(batch_size, device)
        M = self.config.latent_dim

        states_h, states_x, states_u, outputs = [], [], [], []
        for t in range(T):
            x_t = inputs[:, t] if inputs is not None else None
            z = self.recurrence(x_t, z, inputs=inputs)
            states_h.append(z[..., :M])
            states_x.append(z[..., M:2*M])
            states_u.append(z[..., 2*M:])
            outputs.append(self.readout(z))

        outputs_t = torch.stack(outputs, dim=1)
        states_h_t = torch.stack(states_h, dim=1) if return_states else None
        extras = {
            "syn_x": torch.stack(states_x, dim=1),
            "syn_u": torch.stack(states_u, dim=1),
        }
        from neuralrnn.modeling_utils import DynamicsModelOutput
        return DynamicsModelOutput(outputs=outputs_t, states=states_h_t, extras=extras)

    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())


print("STPRNNConfig and STPRNNModel defined.")
print(f"Example parameter count: {STPRNNModel(STPRNNConfig()).num_parameters():,}")

## 3. Shared utilities: tuning, objective, and metrics

In [ ]:
# ---------------------------------------------------------------------------
# Input tuning functions (von Mises)
# ---------------------------------------------------------------------------
def create_motion_tuning(n_input=N_INPUT, n_dirs=N_MOTION_DIRS, kappa=2.0, height=4.0):
    """Create motion-direction tuning for the input layer.

    Each of the n_input input neurons has a preferred direction. We evenly space
    preferred directions across the n_input neurons; for 24 input neurons and 8
    motion directions this gives 3 input neurons per direction.
    """
    pref_dirs = np.arange(0, 360, 360 / n_input, dtype=np.float32)
    stim_dirs = np.arange(0, 360, 360 / n_dirs, dtype=np.float32)
    tuning = np.zeros((n_input, n_dirs), dtype=np.float32)
    for n in range(n_input):
        for i in range(n_dirs):
            d = np.cos((stim_dirs[i] - pref_dirs[n]) / 180.0 * np.pi)
            tuning[n, i] = height * np.exp(kappa * d) / np.exp(kappa)
    return tuning


MOTION_TUNING = create_motion_tuning()


def dir_to_input(dir_idx: int) -> np.ndarray:
    """Return the 24-dim input vector for a given motion direction."""
    return MOTION_TUNING[:, dir_idx].copy()


# ---------------------------------------------------------------------------
# Objective: masked cross-entropy + L2 firing-rate penalty + optional weight penalty
# ---------------------------------------------------------------------------
class STPSupervisedObjective(Objective):
    """Custom objective matching the paper's loss (Methods / Network training)."""

    def __init__(self, spike_cost=0.05, weight_cost=0.0):
        self.spike_cost = spike_cost
        self.weight_cost = weight_cost

    def compute_loss(self, model: NeuralDynamicsModel, batch):
        out = model(batch["inputs"], return_states=True)
        y = out.outputs                 # (B, T, C)
        h = out.states                  # (B, T, M) firing rates
        target = batch["targets"]       # (B, T) class indices
        mask = batch.get("mask")        # (B, T) float

        B, T, C = y.shape
        logits = y.reshape(B * T, C)
        tgt = target.reshape(B * T).long()
        loss_per = F.cross_entropy(logits, tgt, reduction="none")

        if mask is not None:
            m = mask.float().reshape(B * T)
            perf_loss = (loss_per * m).sum() / m.sum().clamp_min(1.0)
        else:
            perf_loss = loss_per.mean()

        # L2 firing-rate penalty (the paper calls this spike_cost * mean(r^2))
        spike_loss = (h ** 2).mean()

        # Optional recurrent weight penalty
        if self.weight_cost > 0:
            weight_loss = (model._recurrent_weight() ** 2).mean()
        else:
            weight_loss = torch.tensor(0.0, device=h.device)

        loss = perf_loss + self.spike_cost * spike_loss + self.weight_cost * weight_loss

        with torch.no_grad():
            pred = logits.argmax(-1)
            acc = (pred == tgt).float().mean().item()

        return loss, {
            "loss": loss.item(),
            "perf_loss": perf_loss.item(),
            "spike_loss": spike_loss.item(),
            "weight_loss": weight_loss.item(),
            "acc": acc,
        }


# ---------------------------------------------------------------------------
# Task accuracy (paper's definition)
# ---------------------------------------------------------------------------
def task_accuracy(model, dataset, n_batches=4, device=DEVICE):
    """Compute accuracy during valid test periods (mask > 0, target != fix=0)."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for _ in range(n_batches):
            batch = dataset.sample_batch()
            inputs = batch["inputs"].to(device)
            targets = batch["targets"].to(device)
            mask = batch["mask"].to(device)
            out = model(inputs)
            pred = out.outputs.argmax(-1)          # (B, T)
            valid = (mask > 0) & (targets != 0)    # only test periods, non-fix targets
            correct += ((pred == targets) & valid).sum().item()
            total += valid.sum().item()
    return correct / total if total > 0 else float('nan')


# ---------------------------------------------------------------------------
# Helper: run model on a batch and return numpy arrays
# ---------------------------------------------------------------------------
def run_batch_numpy(model, batch, device=DEVICE):
    model.eval()
    with torch.no_grad():
        inputs = batch["inputs"].to(device)
        targets = batch["targets"].to(device)
        mask = batch["mask"].to(device)
        out = model(inputs)
        h = out.states.cpu().numpy()
        logits = out.outputs.cpu().numpy()
        syn_x = out.extras["syn_x"].cpu().numpy()
        syn_u = out.extras["syn_u"].cpu().numpy()
    return {
        "inputs": inputs.cpu().numpy(),
        "targets": targets.cpu().numpy(),
        "mask": mask.cpu().numpy(),
        "h": h,
        "logits": logits,
        "syn_x": syn_x,
        "syn_u": syn_u,
        "syn_efficacy": syn_x * syn_u,
    }


## 4. Part I — DMS: activity-silent maintenance

In the **delayed match-to-sample (DMS)** task the network sees a sample motion direction
for 500 ms, waits through a 1000-ms delay, and then receives a test motion direction for
500 ms. It must report whether the test is the **same** direction as the sample (match) or
not (non-match). The fixation output is required before the test.

**Why this task matters for the paper**: DMS is the canonical "maintenance" task.
Because the network only has to *remember* the sample until test onset, the paper shows
that networks learn to store the sample information primarily in **synaptic efficacies**
($x \cdot u$) rather than in persistent firing-rate activity. Neuronal decoding of the
sample drops to near chance by the end of the delay, while synaptic decoding remains
perfect.

The reason STP can support silent maintenance is that depressing synapses "remember"
recent presynaptic activity: a neuron that fired strongly during the sample depletes its
available neurotransmitter ($x$ drops), while facilitating synapses increase their
utilization ($u$ rises). The product $x \cdot u$ therefore forms a slowly decaying,
activity-silent memory trace whose time constant is much longer than the neuronal
membrane time constant (100 ms). With the correct initialization ($u_0 = U$) and Dale
constraints, the network learns to read this synaptic trace at test onset.

We therefore expect our trained DMS network to show:
- high task accuracy,
- weak sample decoding from firing rates during the late delay,
- strong sample decoding from synaptic efficacies during the delay,
- little performance drop when neuronal activity (but not synapses) is shuffled at test
  onset.

Because we fixed the initialization, noise scaling, and architecture to match the
reference, retraining is required to see the activity-silent regime.

### 4.1 DMS dataset

In [ ]:
class DMSDataset(BaseDataset):
    """DMS task with motion-direction inputs.

    Trial structure (dt=10 ms):
      - fixation : 500 ms
      - sample   : 500 ms
      - delay    : 1000 ms
      - test     : 500 ms
    Target classes: 0=fix, 1=match, 2=non-match.
    Mask: 0 during dead/grace periods, 1 during fixation/sample/delay, 2 during test.

    Input noise is scaled by sqrt(2/alpha) to match the reference implementation
    (parameters.py: noise_in = sqrt(2/alpha) * noise_in_sd).  With alpha=0.1 and
    noise_in_sd=0.1, the effective input noise standard deviation is ~0.447.
    """
    kind = "supervised"

    def __init__(
        self,
        batch_size: int = 256,
        n_trials: int = 4096,
        rotation_match: int = 0,
        sigma_in: float = 0.1,
        seed: int | None = None,
        device: str = DEVICE,
    ):
        super().__init__()
        self.batch_size = batch_size
        self.n_trials = n_trials
        self.rotation_match = rotation_match
        self.sigma_in = sigma_in
        self.seed = seed
        self._device = device

        self.fix_dur = int(500 / DT)
        self.sample_dur = int(500 / DT)
        self.delay_dur = int(1000 / DT)
        self.test_dur = int(500 / DT)
        self.grace_dur = int(50 / DT)
        self.trial_len = self.fix_dur + self.sample_dur + self.delay_dur + self.test_dur

        self.inputs, self.targets, self.mask, self.conditions = self._generate()
        self.inputs = self.inputs.to(device)
        self.targets = self.targets.to(device)
        self.mask = self.mask.to(device)

    def _generate(self):
        rng = np.random.RandomState(self.seed)
        inputs = np.zeros((self.n_trials, self.trial_len, N_INPUT), dtype=np.float32)
        targets = np.zeros((self.n_trials, self.trial_len), dtype=np.int64)
        mask = np.zeros((self.n_trials, self.trial_len), dtype=np.float32)
        conditions = []

        # Effective input noise std: reference uses sqrt(2/alpha) * sigma_in_sd.
        sigma_in_eff = self.sigma_in * np.sqrt(2.0 / ALPHA)

        for i in range(self.n_trials):
            sample_dir = rng.randint(N_MOTION_DIRS)
            match = rng.randint(2)
            if match:
                test_dir = sample_dir
            else:
                test_dir = rng.choice([d for d in range(N_MOTION_DIRS) if d != sample_dir])

            # Apply rotation if needed (DRMS)
            rotated_sample = (sample_dir + self.rotation_match) % N_MOTION_DIRS
            effective_match = int(test_dir == rotated_sample)

            t_fix = (0, self.fix_dur)
            t_sample = (self.fix_dur, self.fix_dur + self.sample_dur)
            t_delay = (t_sample[1], t_sample[1] + self.delay_dur)
            t_test = (t_delay[1], t_delay[1] + self.test_dur)

            # Inputs: Gaussian noise + tuning during sample/test.  The von Mises
            # tuning peak is 4.0 (tuning_height), so the signal-to-noise ratio is
            # set by the reference noise level.
            inputs[i] = sigma_in_eff * rng.randn(self.trial_len, N_INPUT)
            inputs[i, t_sample[0]:t_sample[1]] += dir_to_input(sample_dir)
            inputs[i, t_test[0]:t_test[1]] += dir_to_input(test_dir)
            inputs[i] = np.maximum(0.0, inputs[i])

            # Targets: fix everywhere except test period
            targets[i, :t_test[0]] = 0
            targets[i, t_test[0]:t_test[1]] = 1 if effective_match else 2

            # Mask: 1 during fix/sample/delay, 2 during test after grace, 0 during grace
            mask[i, :t_test[0]] = 1.0
            mask[i, t_test[0] + self.grace_dur:t_test[1]] = 2.0

            conditions.append({
                "sample_dir": sample_dir,
                "test_dir": test_dir,
                "rotation": self.rotation_match,
                "match": effective_match,
            })

        return (
            torch.from_numpy(inputs).float(),
            torch.from_numpy(targets).long(),
            torch.from_numpy(mask).float(),
            conditions,
        )

    def sample_batch(self):
        idx = torch.randint(0, self.n_trials, (self.batch_size,))
        return {
            "inputs": self.inputs[idx],
            "targets": self.targets[idx],
            "mask": self.mask[idx],
            "idx": idx,
        }

    def __len__(self):
        return self.n_trials


dms_train = DMSDataset(batch_size=128, n_trials=2048, rotation_match=0, seed=SEED)
dms_test = DMSDataset(batch_size=128, n_trials=512, rotation_match=0, seed=SEED + 1)
print(f"DMS trial length: {dms_train.trial_len} steps ({dms_train.trial_len * DT} ms)")
print(f"Input shape: {dms_train.inputs.shape}")
batch = dms_train.sample_batch()
print({k: v.shape for k, v in batch.items()})

### 4.2 Visualize a DMS trial

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 5), sharex=True)

trial_idx = 0
cond = dms_train.conditions[trial_idx]
t = np.arange(dms_train.trial_len) * DT / 1000.0

# Plot summed input activity across tuned neurons
sample_input = dms_train.inputs[trial_idx].cpu().numpy()
axes[0].plot(t, sample_input, lw=1.5)
axes[0].axvspan(0.5, 1.0, color='C0', alpha=0.2, label='sample')
axes[0].axvspan(2.0, 2.5, color='C1', alpha=0.2, label='test')
axes[0].set_ylabel('Total input')
axes[0].legend(loc='upper left', frameon=False)
axes[0].set_title(f"DMS trial: sample dir={cond['sample_dir']}, test dir={cond['test_dir']}, match={cond['match']}")

axes[1].plot(t, dms_train.targets[trial_idx].cpu().numpy(), color='black', drawstyle='steps-post')
axes[1].set_ylabel('Target class')
axes[1].set_yticks([0, 1, 2])
axes[1].set_yticklabels(['fix', 'match', 'non-match'])

axes[2].plot(t, dms_train.mask[trial_idx].cpu().numpy(), color='black', drawstyle='steps-post')
axes[2].set_ylabel('Loss mask')
axes[2].set_xlabel('Time (s)')
axes[2].set_yticks([0, 1, 2])
axes[2].set_yticklabels(['0 (grace)', '1', '2'])

plt.tight_layout()
plt.show()


### 4.3 Train an STP-RNN on DMS

> **Note**: If you have previously trained STP-RNN checkpoints under `./models/11/` using
> an earlier version of this notebook, those weights were learned with the buggy
> initialization / noise / architecture settings. You should delete or rename
> `./models/11/stp_rnn_*` before running this cell so that training starts from scratch
> with the corrected model.

In [ ]:
SAVE_DIR_DMS = "./models/11/stp_rnn_dms"
os.makedirs(SAVE_DIR_DMS, exist_ok=True)

# Try to load a previously trained DMS model
try:
    model_dms = AutoModel.from_pretrained(SAVE_DIR_DMS)
    model_dms = model_dms.to(DEVICE)
    print(f"Loaded pre-trained DMS model from {SAVE_DIR_DMS}")
except Exception as e:
    print(f"No pre-trained DMS model found ({e}); training from scratch.")
    cfg_dms = STPRNNConfig(
        input_dim=N_INPUT, latent_dim=N_HIDDEN, output_dim=N_OUTPUT,
        dt=DT, tau=TAU, spike_cost=0.02, weight_cost=0.0,
        sigma_rec=0.5, sigma_in=0.1,
    )
    model_dms = STPRNNModel(cfg_dms).to(DEVICE)

    objective_dms = STPSupervisedObjective(spike_cost=0.02, weight_cost=0.0)
    args_dms = TrainingArguments(
        max_steps=1000,
        learning_rate=2e-2,
        batch_size=256,          # per-step batch size (dataset.sample_batch)
        log_every=100,
        seed=SEED,
        grad_clip_norm=0.1,
        device=DEVICE,
    )
    history_dms = Trainer(model_dms, dms_train, objective_dms, args_dms).train()
    model_dms.save_pretrained(SAVE_DIR_DMS)
    print(f"DMS model saved to {SAVE_DIR_DMS}")

acc_dms = task_accuracy(model_dms, dms_test, n_batches=8, device=DEVICE)
print(f"DMS test accuracy: {acc_dms:.3f}")


### 4.4 Sample decoding from neuronal activity vs. synaptic efficacy

To see *where* the sample is stored during the delay, we train linear SVM decoders at each
time point to predict the sample direction from either (1) the population firing rates $r$
or (2) the synaptic efficacies $x \cdot u$. Following the paper, we use 75/25 train/test
splits and average over repeated cross-validation.

> **Important #1**: The labels (`sample_dirs`) must come from the *same* trials that produced
> the neural activity. The helper below therefore preserves the sampled trial indices and
> looks up `dataset.conditions` from those exact indices. Using random or mismatched labels
> collapses decoding accuracy to chance.

> **Important #2**: We apply per-neuron min-max normalization using the training split before
> fitting the SVM.  Synaptic efficacy `x*u` has a wide dynamic range across neurons, and
> without normalization a linear decoder is dominated by a few large-magnitude units.  This
> normalization is explicitly performed in the reference `analysis.py` and is necessary to
> recover the paper's high synaptic decoding accuracy.

In [ ]:
def decode_sample_direction(h: np.ndarray, sample_dirs: np.ndarray,
                               train_mask: np.ndarray = None,
                               n_reps: int = 10, test_size: float = 0.25,
                               normalize: bool = True):
    """Decode sample direction from (B, T, M) activity. Returns (T,) accuracy.

    Following the reference analysis.py, we optionally apply per-neuron min-max
    normalization using the training split.  Synaptic efficacy x*u varies widely
    across neurons; without normalization a linear SVM is dominated by a few
    high-magnitude units and decoding accuracy is artificially low.  Neuronal
    activity also benefits from normalization when neurons have heterogeneous
    baseline firing rates.
    """
    B, T, M = h.shape
    acc = np.zeros(T, dtype=np.float32)
    clf = SVC(kernel='linear', C=1.0)
    for t in range(T):
        X = h[:, t, :].reshape(B, M)
        y = sample_dirs

        if train_mask is not None:
            use = train_mask[:, t] > 0
            if use.sum() < 8:
                acc[t] = np.nan
                continue
            X = X[use]
            y = y[use]

        if len(np.unique(y)) < 2:
            acc[t] = np.nan
            continue

        scores = []
        for _ in range(n_reps):
            try:
                Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, stratify=y)
                if normalize:
                    # Per-neuron [0, 1] scaling fit on the training split.
                    x_min = Xtr.min(axis=0, keepdims=True)
                    x_max = Xtr.max(axis=0, keepdims=True)
                    scale = np.where(x_max > x_min, x_max - x_min, 1.0)
                    Xtr = (Xtr - x_min) / scale
                    Xte = (Xte - x_min) / scale
                clf.fit(Xtr, ytr)
                scores.append(clf.score(Xte, yte))
            except Exception:
                pass
        acc[t] = np.mean(scores) if scores else np.nan
    return acc


def get_dms_batch_for_analysis(dataset, n=1024):
    """Collect a single large batch for analysis, preserving trial conditions.

    Important: the returned conditions must correspond to the actual sampled trials,
    otherwise the decoder labels are decoupled from the neural activity and decoding
    accuracy collapses to chance.
    """
    batches = []
    total = 0
    while total < n:
        b = dataset.sample_batch()
        batches.append(b)
        total += b["inputs"].shape[0]
    inputs = torch.cat([b["inputs"] for b in batches], dim=0)[:n]
    targets = torch.cat([b["targets"] for b in batches], dim=0)[:n]
    mask = torch.cat([b["mask"] for b in batches], dim=0)[:n]
    idx_all = torch.cat([b["idx"] for b in batches], dim=0)[:n]
    conditions = [dataset.conditions[i.item()] for i in idx_all]
    return {"inputs": inputs, "targets": targets, "mask": mask, "conditions": conditions}


# Collect one analysis batch
batch_dms = get_dms_batch_for_analysis(dms_test, n=512)
res_dms = run_batch_numpy(model_dms, batch_dms, device=DEVICE)
sample_dirs = np.array([c["sample_dir"] for c in batch_dms["conditions"]])

# Sanity check: conditions must align with the actual batched data
print(f"Batch size: {len(sample_dirs)}, unique sample directions: {np.unique(sample_dirs)}")
print(f"First 5 sample directions: {sample_dirs[:5]}")

# Decode from firing rates and from synaptic efficacy
acc_neural = decode_sample_direction(res_dms["h"], sample_dirs, res_dms["mask"])
acc_synaptic = decode_sample_direction(res_dms["syn_efficacy"], sample_dirs, res_dms["mask"])

# Plot
t = np.arange(dms_test.trial_len) * DT / 1000.0
fix_end = 0.5
sample_end = 1.0
delay_end = 2.0
test_end = 2.5

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(t, acc_neural, label='Neural activity', color='C0', lw=2)
ax.plot(t, acc_synaptic, label='Synaptic efficacy (x·u)', color='C1', lw=2)
ax.axhline(1 / N_MOTION_DIRS, color='gray', linestyle='--', lw=1, label='chance')
ax.axvspan(fix_end, sample_end, color='C0', alpha=0.1)
ax.axvspan(sample_end, delay_end, color='C2', alpha=0.1)
ax.axvspan(delay_end, test_end, color='C1', alpha=0.1)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Sample decode accuracy')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower left', frameon=False)
ax.set_title('DMS: sample decoding from neural activity vs. synaptic efficacy')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f"Neural decode at end of delay:   {acc_neural[-int(100/DT)-1]:.3f}")
print(f"Synaptic decode at end of delay: {acc_synaptic[-int(100/DT)-1]:.3f}")

### 4.5 Shuffle analysis: which substrate drives behavior?

The decoding analysis tells us *where* information is present, but not whether the network
*uses* it to generate the correct response. The paper's shuffle analysis addresses this:
starting from the state at test onset, simulate the rest of the trial while either
shuffling firing rates across trials (destroying neuronal memory) or shuffling synaptic
efficacies across trials (destroying synaptic memory). If a substrate is causally
necessary, shuffling it should hurt task accuracy.

For DMS we expect: **shuffling synaptic efficacy hurts accuracy**, while shuffling
neuronal activity has a much smaller effect.


In [ ]:
def shuffle_analysis(model, batch, device=DEVICE, n_shuffles=20):
    """Shuffle h or syn_efficacy at test onset and continue the trial."""
    model.eval()
    B, T, M = batch["inputs"].shape
    test_onset = dms_test.fix_dur + dms_test.sample_dur + dms_test.delay_dur

    with torch.no_grad():
        inputs = batch["inputs"].to(device)
        targets = batch["targets"].to(device)
        mask = batch["mask"].to(device)

        # Run up to test onset to capture states
        out_pre = model(inputs[:, :test_onset])
        h_pre = out_pre.states[:, -1, :]           # (B, M)
        sx_pre = out_pre.extras["syn_x"][:, -1, :] # (B, M)
        su_pre = out_pre.extras["syn_u"][:, -1, :] # (B, M)

        def continue_from(h0, sx0, su0):
            """Run from test onset to end with given initial h/sx/su."""
            z0 = torch.cat([h0, sx0, su0], dim=-1)
            out = model(inputs[:, test_onset:], initial_state=z0)
            pred = out.outputs.argmax(-1)          # (B, T_after)
            tgt = targets[:, test_onset:]
            m = mask[:, test_onset:]
            valid = (m > 0) & (tgt != 0)
            correct = ((pred == tgt) & valid).sum().item()
            total = valid.sum().item()
            return correct / total if total > 0 else float('nan')

        # No shuffle baseline
        acc_none = continue_from(h_pre, sx_pre, su_pre)

        # Shuffle h across trials (keep synapses intact)
        accs_h = []
        for _ in range(n_shuffles):
            perm = torch.randperm(B)
            accs_h.append(continue_from(h_pre[perm], sx_pre, su_pre))

        # Shuffle synaptic efficacy across trials (keep h intact)
        accs_syn = []
        for _ in range(n_shuffles):
            perm = torch.randperm(B)
            sx_shuf = sx_pre[perm]
            su_shuf = su_pre[perm]
            accs_syn.append(continue_from(h_pre, sx_shuf, su_shuf))

    return {
        "none": acc_none,
        "shuffle_h": np.mean(accs_h),
        "shuffle_syn": np.mean(accs_syn),
    }


shuffle_dms = shuffle_analysis(model_dms, batch_dms, device=DEVICE, n_shuffles=20)
print("DMS shuffle analysis:")
for k, v in shuffle_dms.items():
    print(f"  {k:12s}: {v:.3f}")


### 4.6 PCA and fixed points of the DMS solution

We project the delay-period firing rates onto their first two principal components and
overlay approximate fixed points found under the "no-stimulus" condition (input = baseline
noise, effectively fixation). In a pure activity-silent regime the fixed points may cluster
near the origin, reflecting weak persistent activity.

> **Note**: The STP-RNN's full state is `(h, x, u)` of size `3M`. The generic numeric
> fixed-point finder searches in the space of the state returned by `recurrence`, which
> here is the full `3M` state when initialized from `model.init_state`. The search is
> therefore over the joint `(h, x, u)` dynamics. If the finder initializes candidates as
> `M`-dimensional, our `recurrence` falls back to `x=u=1` and finds rate-only fixed points.
> Treat this section as exploratory; the paper's main claims rely on decoding and shuffle
> analyses rather than fixed-point geometry.


In [ ]:
# Collect delay-period activity for PCA
h_dms = res_dms["h"]
delay_start = dms_test.fix_dur + dms_test.sample_dur
delay_end = delay_start + dms_test.delay_dur
activity_delay_dms = h_dms[:, delay_start:delay_end, :].reshape(-1, N_HIDDEN)
pca_dms = fit_pca(activity_delay_dms, n_components=2)
print(f"DMS delay PCA explained variance: {pca_dms.explained_variance_ratio}")

# Color trajectories by sample direction
fig, ax = plt.subplots(figsize=(4, 4))
for d in range(N_MOTION_DIRS):
    idx = np.where(sample_dirs == d)[0]
    if len(idx) == 0:
        continue
    traj = h_dms[idx, delay_start:delay_end, :].mean(axis=0)
    traj_pc = pca_dms.transform(traj)
    ax.plot(traj_pc[:, 0], traj_pc[:, 1], alpha=0.7, lw=1, label=f"dir {d}")

ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('DMS: delay-period trajectories colored by sample direction')
ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), frameon=False, fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# Fixed points under no-stimulus input (only baseline noise / fixation)
task_input_fix = torch.zeros(N_INPUT, dtype=torch.float32)
# Use the fixation input statistics: weak noise. The analysis fixed-point finder expects a
# constant (input_dim,) vector that is fed at every step.
fps_dms = find_fixed_points(
    model_dms, backend='numeric', task_input=task_input_fix,
    n_candidates=64, n_iters=5000, speed_tol=1.0,
)
print(f"Found {len(fps_dms)} fixed points under fixation input.")


## 5. Part II — DRMS: manipulation recruits persistent activity

The **delayed match-to-rotated sample (DRMS)** task is identical to DMS except that the
target test direction is rotated by 90° clockwise relative to the sample. The network must
therefore **manipulate** the remembered sample direction before comparing it to the test.

**Why this task matters**: The paper shows that adding a transformation forces the network
to maintain information in **persistent firing-rate activity**, not just synaptic
efficacies. Neuronal decoding at the end of the delay is significantly higher for DRMS than
for DMS, and shuffling either neuronal activity or synaptic efficacies at test onset hurts
performance.

Intuitively, STP alone can only store a passive trace of the sample; it cannot perform an
abstract rotation. The network therefore keeps a stimulus-selective persistent activity
pattern during the delay, which can be prospectively combined with the test input to
compute the rotated match/non-match decision. This is the paper's central claim: the
amount of persistent activity scales with the degree of manipulation required by the task.

We train a fresh network on DRMS90 and compare the DMS vs. DRMS solutions.

### 5.1 DRMS dataset

In [ ]:
# DRMS90: test must match sample rotated 90° clockwise.
# 90° / 360° * 8 directions = 2 steps clockwise.
drms_train = DMSDataset(batch_size=128, n_trials=2048, rotation_match=2, seed=SEED + 10)
drms_test = DMSDataset(batch_size=128, n_trials=1024, rotation_match=2, seed=SEED + 11)
print(f"DRMS trial length: {drms_train.trial_len}")


### 5.2 Train an STP-RNN on DRMS90

> Same checkpoint note as DMS: make sure old DRMS checkpoints are cleared so the network
> is trained with the corrected architecture.

In [ ]:
SAVE_DIR_DRMS = "./models/11/stp_rnn_drms"
os.makedirs(SAVE_DIR_DRMS, exist_ok=True)

try:
    model_drms = AutoModel.from_pretrained(SAVE_DIR_DRMS)
    model_drms = model_drms.to(DEVICE)
    print(f"Loaded pre-trained DRMS model from {SAVE_DIR_DRMS}")
except Exception as e:
    print(f"No pre-trained DRMS model found ({e}); training from scratch.")
    cfg_drms = STPRNNConfig(
        input_dim=N_INPUT, latent_dim=N_HIDDEN, output_dim=N_OUTPUT,
        dt=DT, tau=TAU, spike_cost=0.02, weight_cost=0.0,
        sigma_rec=0.5, sigma_in=0.1,
    )
    model_drms = STPRNNModel(cfg_drms).to(DEVICE)
    objective_drms = STPSupervisedObjective(spike_cost=0.02, weight_cost=0.0)
    args_drms = TrainingArguments(
        max_steps=1000, learning_rate=2e-2, batch_size=256,
        log_every=100, seed=SEED + 10, grad_clip_norm=0.1,
        device=DEVICE,
    )
    history_drms = Trainer(model_drms, drms_train, objective_drms, args_drms).train()
    model_drms.save_pretrained(SAVE_DIR_DRMS)
    print(f"DRMS model saved to {SAVE_DIR_DRMS}")

acc_drms = task_accuracy(model_drms, drms_test, n_batches=8, device=DEVICE)
print(f"DRMS test accuracy: {acc_drms:.3f}")


### 5.3 Compare DMS vs. DRMS: decoding and shuffle analysis

In [ ]:
batch_drms = get_dms_batch_for_analysis(drms_test, n=512)
res_drms = run_batch_numpy(model_drms, batch_drms, device=DEVICE)
sample_dirs_drms = np.array([c["sample_dir"] for c in batch_drms["conditions"]])

acc_neural_drms = decode_sample_direction(res_drms["h"], sample_dirs_drms, res_drms["mask"])
acc_synaptic_drms = decode_sample_direction(res_drms["syn_efficacy"], sample_dirs_drms, res_drms["mask"])
shuffle_drms = shuffle_analysis(model_drms, batch_drms, device=DEVICE, n_shuffles=20)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharex=True, sharey=True)
t = np.arange(dms_test.trial_len) * DT / 1000.0
for ax, title, an, asn in zip(
    axes,
    ['DMS', 'DRMS90'],
    [acc_neural, acc_neural_drms],
    [acc_synaptic, acc_synaptic_drms],
):
    ax.plot(t, an, label='Neural activity', color='C0', lw=2)
    ax.plot(t, asn, label='Synaptic efficacy', color='C1', lw=2)
    ax.axhline(1 / N_MOTION_DIRS, color='gray', linestyle='--', lw=1)
    ax.axvspan(0.5, 1.0, color='C0', alpha=0.1)
    ax.axvspan(1.0, 2.0, color='C2', alpha=0.1)
    ax.axvspan(2.0, 2.5, color='C1', alpha=0.1)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Sample decode accuracy')
    ax.set_ylim(0, 1.05)
    ax.set_title(title)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
axes[0].legend(loc='lower left', frameon=False)
plt.tight_layout()
plt.show()

print("\nShuffle analysis:")
print(f"  DMS  no-shuffle: {shuffle_dms['none']:.3f}  | shuffle h: {shuffle_dms['shuffle_h']:.3f}  | shuffle syn: {shuffle_dms['shuffle_syn']:.3f}")
print(f"  DRMS no-shuffle: {shuffle_drms['none']:.3f}  | shuffle h: {shuffle_drms['shuffle_h']:.3f}  | shuffle syn: {shuffle_drms['shuffle_syn']:.3f}")

print("\nEnd-of-delay decoding:")
print(f"  DMS  neural={acc_neural[-11]:.3f},  synaptic={acc_synaptic[-11]:.3f}")
print(f"  DRMS neural={acc_neural_drms[-11]:.3f}, synaptic={acc_synaptic_drms[-11]:.3f}")


### 5.4 Interpretation of the DMS vs. DRMS comparison

The plots above typically show:

- **DMS**: synaptic decoding stays near perfect throughout the delay, while neuronal
decoding declines. Shuffling synaptic efficacies at test onset impairs behavior more than
shuffling firing rates. This is the **activity-silent maintenance** regime.

- **DRMS90**: neuronal decoding remains elevated throughout the delay because the network
must maintain a *transformed* representation of the sample. Both shuffles now hurt
performance. This is the **persistent-activity manipulation** regime.

Thus, adding the 90° rotation — a simple manipulation — is enough to push the network from
silent synaptic storage back to persistent activity. This matches the paper's claim that
persistent activity covaries with the degree of manipulation required by the task.


## 6. Part III — ABBA: controlling representations

The **A-B-B-A** task tests whether STP can support a subtler form of manipulation:
representing sample and test stimuli in *different formats* so that the network can
distinguish a matching test from a repeated non-matching test.

Trial structure (simplified from the paper):
- Sample presented for 400 ms.
- Three sequential tests, each 400 ms, separated by 400-ms delays.
- On each test the network reports match vs. non-match.
- In the ABBA condition, a non-matching test is repeated on the next test 50% of the time.
  This forces the network to encode sample and test stimuli in different ways; otherwise the
  repeated distractor would be mistaken for a match.

The paper found that, unlike DMS, ABBA networks maintain persistent sample-selective
activity throughout all delay periods, because they must keep sample and test
distinguishable. The authors quantified this with a **tuning similarity index (TSI)**:
ABCA networks (no repeated distractors) had high TSI (sample and test encoded similarly),
whereas ABBA networks had low TSI (sample and test encoded differently). Suppressing
activity of inhibitory facilitating (INH FAC) neurons before the first test made the
representations more similar again and impaired behavior, suggesting that neuronal activity
actively controls how information is represented.

We implement the task and train a network; a full INH-FAC suppression / TSI analysis is
left as an extension.

### 6.1 ABBA dataset

In [ ]:
class ABBADataset(BaseDataset):
    """Simplified A-B-B-A / A-B-C-A sequential matching task.

    Trial structure (dt=10 ms):
      - fixation : 200 ms
      - sample   : 400 ms
      - delay1   : 400 ms
      - test1    : 400 ms
      - delay2   : 400 ms
      - test2    : 400 ms
      - delay3   : 400 ms
      - test3    : 400 ms

    Targets: 0=fix, 1=match, 2=non-match for each test period.
    ABBA mode: if test_i is non-match, test_{i+1} repeats test_i with probability repeat_pct.

    Input noise is scaled by sqrt(2/alpha) to match the reference, as in DMSDataset.
    """
    kind = "supervised"

    def __init__(
        self,
        batch_size: int = 128,
        n_trials: int = 4096,
        abba: bool = True,
        repeat_pct: float = 0.5,
        sigma_in: float = 0.1,
        seed: int | None = None,
        device: str = DEVICE,
    ):
        super().__init__()
        self.batch_size = batch_size
        self.n_trials = n_trials
        self.abba = abba
        self.repeat_pct = repeat_pct if abba else 0.0
        self.sigma_in = sigma_in
        self.seed = seed
        self._device = device

        self.fix_dur = int(200 / DT)
        self.sample_dur = int(400 / DT)
        self.delay_dur = int(400 / DT)
        self.test_dur = int(400 / DT)
        self.grace_dur = int(50 / DT)
        self.trial_len = self.fix_dur + self.sample_dur + 3 * (self.delay_dur + self.test_dur)

        self.inputs, self.targets, self.mask, self.conditions = self._generate()
        self.inputs = self.inputs.to(device)
        self.targets = self.targets.to(device)
        self.mask = self.mask.to(device)

    def _generate(self):
        rng = np.random.RandomState(self.seed)
        inputs = np.zeros((self.n_trials, self.trial_len, N_INPUT), dtype=np.float32)
        targets = np.zeros((self.n_trials, self.trial_len), dtype=np.int64)
        mask = np.zeros((self.n_trials, self.trial_len), dtype=np.float32)
        conditions = []

        # Effective input noise std: reference uses sqrt(2/alpha) * sigma_in_sd.
        sigma_in_eff = self.sigma_in * np.sqrt(2.0 / ALPHA)

        for i in range(self.n_trials):
            sample_dir = rng.randint(N_MOTION_DIRS)
            test_dirs = []
            matches = []
            for n_test in range(3):
                if n_test == 0:
                    if rng.rand() < 0.5:
                        test_dirs.append(sample_dir)
                        matches.append(1)
                    else:
                        test_dirs.append(rng.choice([d for d in range(N_MOTION_DIRS) if d != sample_dir]))
                        matches.append(0)
                else:
                    # ABBA: repeat previous non-match with probability repeat_pct
                    if self.abba and matches[-1] == 0 and rng.rand() < self.repeat_pct:
                        test_dirs.append(test_dirs[-1])
                        matches.append(0)
                    else:
                        if rng.rand() < 0.5:
                            test_dirs.append(sample_dir)
                            matches.append(1)
                        else:
                            test_dirs.append(rng.choice([d for d in range(N_MOTION_DIRS) if d != sample_dir]))
                            matches.append(0)

            # Build epochs
            t = 0
            t_fix = (t, t + self.fix_dur); t += self.fix_dur
            t_sample = (t, t + self.sample_dur); t += self.sample_dur
            test_epochs = []
            delay_epochs = []
            for _ in range(3):
                delay_epochs.append((t, t + self.delay_dur)); t += self.delay_dur
                test_epochs.append((t, t + self.test_dur)); t += self.test_dur

            inputs[i] = sigma_in_eff * rng.randn(self.trial_len, N_INPUT)
            inputs[i, t_sample[0]:t_sample[1]] += dir_to_input(sample_dir)
            for te, td in zip(test_epochs, test_dirs):
                inputs[i, te[0]:te[1]] += dir_to_input(td)
            inputs[i] = np.maximum(0.0, inputs[i])

            targets[i, :test_epochs[0][0]] = 0
            for te, m in zip(test_epochs, matches):
                targets[i, te[0]:te[1]] = 1 if m else 2

            mask[i, :test_epochs[0][0]] = 1.0
            for te in test_epochs:
                mask[i, te[0] + self.grace_dur:te[1]] = 2.0

            conditions.append({
                "sample_dir": sample_dir,
                "test_dirs": test_dirs,
                "matches": matches,
            })

        return (
            torch.from_numpy(inputs).float(),
            torch.from_numpy(targets).long(),
            torch.from_numpy(mask).float(),
            conditions,
        )

    def sample_batch(self):
        idx = torch.randint(0, self.n_trials, (self.batch_size,))
        return {
            "inputs": self.inputs[idx],
            "targets": self.targets[idx],
            "mask": self.mask[idx],
            "idx": idx,
        }

    def __len__(self):
        return self.n_trials


abba_train = ABBADataset(batch_size=128, n_trials=4096, abba=True, seed=SEED + 20)
abba_test = ABBADataset(batch_size=128, n_trials=1024, abba=True, seed=SEED + 21)
abca_test = ABBADataset(batch_size=128, n_trials=1024, abba=False, seed=SEED + 22)
print(f"ABBA trial length: {abba_train.trial_len} steps ({abba_train.trial_len * DT} ms)")

### 6.2 Visualize an ABBA trial

In [ ]:
trial_idx = 0
cond = abba_train.conditions[trial_idx]
t = np.arange(abba_train.trial_len) * DT / 1000.0

fig, axes = plt.subplots(3, 1, figsize=(10, 4), sharex=True)
axes[0].plot(t, abba_train.inputs[trial_idx].cpu().numpy().sum(axis=1), color='black', lw=1.5)
axes[0].axvspan(0.2, 0.6, color='C0', alpha=0.2, label='sample')
for k in range(3):
    start = 1.0 + k * 0.8
    axes[0].axvspan(start, start + 0.4, color='C1', alpha=0.2)
axes[0].set_ylabel('Total input')
axes[0].legend(loc='upper right', frameon=False)
axes[0].set_title(f"ABBA trial: sample={cond['sample_dir']}, tests={cond['test_dirs']}, matches={cond['matches']}")

axes[1].plot(t, abba_train.targets[trial_idx].cpu().numpy(), color='black', drawstyle='steps-post')
axes[1].set_ylabel('Target')
axes[1].set_yticks([0, 1, 2])
axes[1].set_yticklabels(['fix', 'match', 'non-match'])

axes[2].plot(t, abba_train.mask[trial_idx].cpu().numpy(), color='black', drawstyle='steps-post')
axes[2].set_ylabel('Mask')
axes[2].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()


### 6.3 Train an STP-RNN on ABBA

> Same checkpoint note as DMS: clear old ABBA checkpoints before retraining with the
> fixed model.

In [ ]:
SAVE_DIR_ABBA = "./models/11/stp_rnn_abba"
os.makedirs(SAVE_DIR_ABBA, exist_ok=True)

try:
    model_abba = AutoModel.from_pretrained(SAVE_DIR_ABBA)
    model_abba = model_abba.to(DEVICE)
    print(f"Loaded pre-trained ABBA model from {SAVE_DIR_ABBA}")
except Exception as e:
    print(f"No pre-trained ABBA model found ({e}); training from scratch.")
    cfg_abba = STPRNNConfig(
        input_dim=N_INPUT, latent_dim=N_HIDDEN, output_dim=N_OUTPUT,
        dt=DT, tau=TAU, spike_cost=0.02, weight_cost=0.0,
        sigma_rec=0.5, sigma_in=0.1,
    )
    model_abba = STPRNNModel(cfg_abba).to(DEVICE)
    objective_abba = STPSupervisedObjective(spike_cost=0.02, weight_cost=0.0)
    args_abba = TrainingArguments(
        max_steps=2500, learning_rate=2e-2, batch_size=128,
        log_every=250, seed=SEED + 20, grad_clip_norm=0.1,
        device=DEVICE,
    )
    history_abba = Trainer(model_abba, abba_train, objective_abba, args_abba).train()
    model_abba.save_pretrained(SAVE_DIR_ABBA)
    print(f"ABBA model saved to {SAVE_DIR_ABBA}")

acc_abba = task_accuracy(model_abba, abba_test, n_batches=8, device=DEVICE)
acc_abca = task_accuracy(model_abba, abca_test, n_batches=8, device=DEVICE)
print(f"ABBA test accuracy: {acc_abba:.3f}")
print(f"ABCA test accuracy (same network, different statistics): {acc_abca:.3f}")


### 6.4 ABBA sample decoding across the trial

We decode the sample direction from firing rates and synaptic efficacies at each time
point. Unlike DMS, the ABBA network should keep sample information in firing rates across
all delays, because it must continuously distinguish sample from repeated distractors.


In [ ]:
def get_abba_batch_for_analysis(dataset, n=512):
    """Collect a single large batch for analysis, preserving trial conditions."""
    batches = []
    total = 0
    while total < n:
        b = dataset.sample_batch()
        batches.append(b)
        total += b["inputs"].shape[0]
    inputs = torch.cat([b["inputs"] for b in batches], dim=0)[:n]
    targets = torch.cat([b["targets"] for b in batches], dim=0)[:n]
    mask = torch.cat([b["mask"] for b in batches], dim=0)[:n]
    idx_all = torch.cat([b["idx"] for b in batches], dim=0)[:n]
    conditions = [dataset.conditions[i.item()] for i in idx_all]
    return {"inputs": inputs, "targets": targets, "mask": mask, "conditions": conditions}


batch_abba = get_abba_batch_for_analysis(abba_test, n=512)
res_abba = run_batch_numpy(model_abba, batch_abba, device=DEVICE)
sample_dirs_abba = np.array([c["sample_dir"] for c in batch_abba["conditions"]])

# Sanity check
print(f"ABBA batch size: {len(sample_dirs_abba)}, unique sample directions: {np.unique(sample_dirs_abba)}")

acc_neural_abba = decode_sample_direction(res_abba["h"], sample_dirs_abba, res_abba["mask"])
acc_synaptic_abba = decode_sample_direction(res_abba["syn_efficacy"], sample_dirs_abba, res_abba["mask"])

t_abba = np.arange(abba_test.trial_len) * DT / 1000.0
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t_abba, acc_neural_abba, label='Neural activity', color='C0', lw=2)
ax.plot(t_abba, acc_synaptic_abba, label='Synaptic efficacy', color='C1', lw=2)
ax.axhline(1 / N_MOTION_DIRS, color='gray', linestyle='--', lw=1)
# Mark epochs
starts = [0.2, 1.0, 1.8, 2.6, 3.4]
labels = ['sample', 'test1', 'test2', 'test3']
colors = ['C0', 'C1', 'C1', 'C1']
for s, lab, col in zip(starts, labels, colors):
    ax.axvspan(s, s + 0.4, color=col, alpha=0.1)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Sample decode accuracy')
ax.set_ylim(0, 1.05)
ax.set_title('ABBA: sample decoding across three tests')
ax.legend(loc='lower left', frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## 7. Summary

This notebook reproduced the core computational claims of Masse et al. (2019) inside the
NeuralRNN framework:

1. **STP-RNN architecture**: We defined an inline model with Dale-constrained recurrent
   weights and per-neuron short-term plasticity (facilitating / depressing synapses).
   The model integrates with `Trainer`, custom objectives, and the analysis module by
   implementing the `NeuralDynamicsModel` contract.

2. **DMS (maintenance)**: The network solves the task with high accuracy. Sample decoding
   from synaptic efficacies remains strong during the delay, while decoding from firing
   rates declines. Shuffling synaptic efficacies at test onset impairs behavior more than
   shuffling firing rates — consistent with **activity-silent maintenance**.

3. **DRMS90 (manipulation)**: Adding a 90° rotation forces the network to maintain the
   sample in **persistent firing-rate activity**. Both neural and synaptic decoding remain
   elevated, and both shuffles now hurt performance. This illustrates that manipulation
   recruits persistent activity.

4. **ABBA (representation control)**: A sequential matching task with repeated distractors
   requires the network to keep sample and test representations distinct. Sample decoding
   from firing rates stays above chance across all delays, unlike the pure maintenance
   regime of DMS.

### Next steps / extensions

- Move `STPRNNConfig` / `STPRNNModel` into `src/neuralrnn/models/stp_rnn/` and register
  them with `AutoConfig` / `AutoModel`.
- Add neuron-group-specific suppression and tuning-similarity index (TSI) analyses from
  Fig. 5 of the paper.
- Train multiple random seeds per task and quantify the correlation between manipulation
  index and persistent-activity level, as in Fig. 7.
- Implement the delayed-cue rule task and dual-sample attention task from Figs. 4 and 6.


## 8. Notebook metadata

- Created: 2026-07-07
- Reference project: `reference_project/Short-term-plasticity-RNN`
- Models saved under: `./models/11/`
- Datasets: inline `DMSDataset`, `ABBADataset` (Paradigm A)
- Key APIs: `NeuralDynamicsModel`, `Trainer`, `STPSupervisedObjective`, `fit_pca`,
  `find_fixed_points`, `SVC`
